In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import math
import random
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

class HEMSEnvFinal(gym.Env):
    metadata = {"render.modes": ["human"]}

    def __init__(self, config):
        super().__init__()
        # Load data
        self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
        self.load_df = pd.read_csv(config["load_path"], parse_dates=["Date_Time"])
        self.solar_df.columns = self.solar_df.columns.str.strip()
        self.load_df.columns = self.load_df.columns.str.strip()

        # Config and hardware
        self.config = config
        self.solar_panel_area = config["solar_panel_area"]
        self.max_battery = float(config["max_battery_kwh"])
        self.appliances = config["appliances"]
        
        self.n_steps = min(len(self.load_df), len(self.solar_df))
        self.dt_hours = 1 / 60.0
        self.steps_per_day = 24 * 60

        # Action space
        self.action_space = spaces.Discrete(2**len(self.appliances))
        
        # Observation space
        num_obs_features = 4 + len(self.appliances)
        obs_low = np.array([-1.0] * num_obs_features, dtype=np.float32)
        obs_high = np.array([1.0] * num_obs_features, dtype=np.float32)
        obs_high[0] = 1e6
        self.observation_space = spaces.Box(obs_low, obs_high, dtype=np.float32)
        
        self.time_step = 0
        self.battery_soc = 0.0
        self._reset_appliance_states()
        self.episode_info = []

    def _reset_appliance_states(self):
        # This function creates the internal state tracker for the appliances
        self.appliance_states = []
        for app in self.appliances:
            self.appliance_states.append({
                "name": app["name"], "is_running": False,
                "steps_remaining": 0, "task_completed_today": False
            })

    def _compute_solar_output_kw(self, solar_row):
        ghi = pd.to_numeric(solar_row.get("ghi_pyr"), errors='coerce')
        humidity = pd.to_numeric(solar_row.get("relative_humidity"), errors='coerce')
        if pd.isna(ghi) or pd.isna(humidity): return 0.0
        panel_efficiency = 0.18
        weather_factor = max(0.0, 1.0 - (humidity / 100.0))
        pv_kw = (ghi * self.solar_panel_area * panel_efficiency * weather_factor) / 1000.0
        return max(pv_kw, 0.0)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.time_step = 0
        self.battery_soc = 0.5 * self.max_battery
        self._reset_appliance_states()
        self.episode_info = []
        obs, info = self._get_obs()
        return obs, info

    def _get_obs(self):
        idx = self.time_step % self.n_steps
        solar_row = self.solar_df.iloc[idx]
        solar_kw = self._compute_solar_output_kw(solar_row)
        soc_norm = self.battery_soc / self.max_battery if self.max_battery > 0 else 0.0
        hour = (self.time_step // 60) % 24
        hour_rad = (2.0 * math.pi * hour) / 24.0
        # *** FIX: Read from the internal state tracker, not the config ***
        appliance_statuses = [1.0 if not s["task_completed_today"] else 0.0 for s in self.appliance_states]
        obs_list = [solar_kw, soc_norm, math.sin(hour_rad), math.cos(hour_rad)] + appliance_statuses
        return np.array(obs_list, dtype=np.float32), {}

    def _get_grid_price_for_hour(self, hour):
        return 46.85 if 19 <= hour <= 23 else 40.53

    def step(self, action):
        idx = self.time_step % self.n_steps
        current_load_row = self.load_df.iloc[idx]
        
        if self.time_step > 0 and self.time_step % self.steps_per_day == 0:
            for state in self.appliance_states:
                state["task_completed_today"] = False

        action_commands = [int(bit) for bit in np.binary_repr(action, width=len(self.appliances))]
        
        appliance_load_kw = 0
        for i, start_command in enumerate(action_commands):
            state = self.appliance_states[i]
            if start_command == 1 and not state["is_running"] and not state["task_completed_today"]:
                state["is_running"] = True
                state["steps_remaining"] = self.appliances[i]["duration_steps"]
                state["task_completed_today"] = True
            
            if state["is_running"]:
                appliance_load_kw += self.appliances[i]["power_kw"]
                state["steps_remaining"] -= 1
                if state["steps_remaining"] == 0:
                    state["is_running"] = False

        base_load_kw = current_load_row['Refrigerator_kW'] + current_load_row['WD_kW'] + current_load_row['Kitchen_light1_kW'] + current_load_row['Kitchen_light2_kW'] + current_load_row['SR_kW']
        total_load_kw = base_load_kw + appliance_load_kw
        total_load_kwh = total_load_kw * self.dt_hours
        
        solar_row = self.solar_df.iloc[idx]
        solar_kwh_available = self._compute_solar_output_kw(solar_row) * self.dt_hours
        
        grid_energy, battery_discharge = 0.0, 0.0
        remaining_load = total_load_kwh
        solar_supplied = min(solar_kwh_available, remaining_load)
        remaining_load -= solar_supplied
        if self.battery_soc > 0 and remaining_load > 0:
            discharge = min(self.battery_soc, remaining_load / 0.95)
            battery_discharge = discharge
            self.battery_soc -= discharge
            remaining_load -= discharge * 0.95
        if remaining_load > 0:
            grid_energy = remaining_load
        if solar_kwh_available > solar_supplied:
            charge = min((solar_kwh_available - solar_supplied) * 0.95, self.max_battery - self.battery_soc)
            self.battery_soc += charge

        hour = (self.time_step // 60) % 24
        grid_cost = grid_energy * self._get_grid_price_for_hour(hour)
        battery_deg_cost = battery_discharge * self.config.get("battery_deg_cost_per_kwh", 0.01)
        
        reward = -(grid_cost + battery_deg_cost)
        
        terminated = False
        missed_task_today = False
        if (self.time_step + 1) % self.steps_per_day == 0:
            for state in self.appliance_states:
                if not state["task_completed_today"]:
                    terminated = True
                    missed_task_today = True
                    break
        
        if not terminated:
            terminated = self.time_step >= (self.n_steps - 1) 

        info = { "total_cost": grid_cost + battery_deg_cost, "task_missed": missed_task_today }
        
        self.time_step += 1
        obs, _ = self._get_obs()
        return obs, reward, terminated, False, info

def analyze_and_define_jobs(data_path):
    df = pd.read_csv(data_path)
    df.columns = df.columns.str.strip()
    df.fillna(0, inplace=True)
    
    schedulable_cols = ['Laundary_kW', 'AC_BR_kW', 'AC_GR_kW','WP_kW']
    
    appliances = []
    for col in schedulable_cols:
        power_draw = df[df[col] > 0.1][col].mean()
        df['running'] = (df[col] > 0.1).astype(int)
        df['block'] = (df['running'].diff() != 0).astype(int).cumsum()
        run_lengths = df[df['running'] == 1].groupby('block').size()
        
        duration = max(30, run_lengths.median()) if not run_lengths.empty else 30
        
        if pd.notna(power_draw) and power_draw > 0.1:
             appliances.append({
                "name": col, "power_kw": power_draw, "duration_steps": int(duration)
            })
    return appliances

if __name__ == "__main__":
    
    appliance_jobs = analyze_and_define_jobs("homee.csv")
    
    env_config = {
        "load_path": "homee.csv",
        "solar_path": "solar.csv", 
        "solar_panel_area": 44.4,
        "max_battery_kwh": 15.0,
        "appliances": appliance_jobs
    }
    
    def make_env():
        return HEMSEnvFinal(env_config)

    print("\n--- Training Final Multi-Task HEMS Agent ---")
    vec_env = DummyVecEnv([make_env for _ in range(4)])
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=False, clip_obs=10.0)
    model = PPO("MlpPolicy", vec_env, policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])), verbose=0)
    model.learn(total_timesteps=300000, progress_bar=True)
    
    print("\n--- Evaluating Final Multi-Task HEMS Agent ---")
    all_infos = []
    obs = vec_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, info = vec_env.step(action)
        all_infos.append(info[0])
        done = terminated[0]

    results = pd.DataFrame(all_infos)
    
    total_cost = results['total_cost'].sum()
    missed_tasks = results['task_missed'].sum()
    num_days = len(results) / (24 * 60)

    print("\n\n==============================================")
    print("===   Definitive HEMS Scheduling Summary   ===")
    print("==============================================")
    print(f"Total Simulation Days: {num_days:.0f}")
    print(f"Total Annual Energy Cost: {total_cost:,.2f} PKR")
    print(f"Total Days with Missed Tasks: {missed_tasks:.0f}")


--- Training Final Multi-Task HEMS Agent ---



--- Evaluating Final Multi-Task HEMS Agent ---


===   Definitive HEMS Scheduling Summary   ===
Total Simulation Days: 92
Total Annual Energy Cost: 17,238.94 PKR
Total Days with Missed Tasks: 0
